##Step1:Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from delta.tables import *
from delta.tables import DeltaTable


##Step2:Read Fact Tables

In [0]:
fact_orders = spark.table("fact_orders")

###Gold Table 2 : gold_daily_sales

In [0]:
gold_daily_sales=fact_orders\
    .withColumn("order_date", to_date("order_ts"))\
    .groupBy("order_date")\
        .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"),2).alias("total_sales")
        )\
            .orderBy("order_date")
            

In [0]:
gold_daily_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_daily_sales")

In [0]:
display(gold_daily_sales.limit(10))

order_date,total_orders,total_quantity,total_sales
2025-10-01,60,193,7931198.78
2025-10-02,49,135,5833780.93
2025-10-03,57,160,6742787.45
2025-10-04,63,201,8764260.0
2025-10-05,52,159,6296934.8
2025-10-06,61,188,8034282.17
2025-10-07,63,174,8086661.39
2025-10-08,63,185,6900786.52
2025-10-09,49,136,6025594.72
2025-10-10,65,208,7819608.32


###Gold Table 2 : gold_category_sales

In [0]:
dim_product = spark.table("dim_products_scd2")

In [0]:
dim_product=dim_product.filter(
    col("is_current")=="true"
)\
.select(
    "product_sk",
    "category"
)

In [0]:
gold_category_sales=fact_orders\
    .join(dim_product
          ,"product_sk",
          "inner"
          )\
    .groupBy("category")\
        .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"),2).alias("total_sales")
    )\
    .orderBy(desc("total_sales"))



In [0]:
gold_category_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_category_sales")

In [0]:
display(gold_category_sales.limit(10))

category,total_orders,total_quantity,total_sales
Home,2564,7701,3.1474037014E8
Fashion,2423,7260,2.9833187873E8
Electronics,2104,6256,2.5726551771E8
Grocery,2008,5897,2.4310422515E8
Beauty,1946,5762,2.286439617E8
Unknown,138,419,1.753457823E7
null,51,158,5916422.92


###Gold Table 3 : gold_segment_sales

In [0]:
dim_customer = spark.table("dim_customers_scd2")\
    .filter("is_current = true")\
    .select(
        "customer_sk",
        "segment"
    )


In [0]:
gold_segment_sales =fact_orders\
    .join(
        dim_customer,
        "customer_sk"
    )\
    .groupBy("segment")\
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"),2).alias("total_sales"),
        round(avg("gross_amount"),2).alias("avg_order_value")
    )\
    .orderBy(desc("total_sales"))


In [0]:
gold_segment_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_segment_sales")

In [0]:
display(gold_segment_sales.limit(10))

segment,total_orders,total_quantity,total_sales,avg_order_value
Platinum,2954,8710,3.5613828575E8,120561.37
Silver,2908,8578,3.5071965353E8,120605.11
Regular,2832,8466,3.4741735279E8,122675.62
Gold,2682,8081,3.2665684081E8,121795.99


###Gold Table 4 : gold_region_sales

In [0]:
dim_store = spark.table("silver_stores")\
    .select(
        "store_id",
        "region"
    )


In [0]:
gold_region_sales =fact_orders\
    .join(
        dim_store,
        "store_id"
    )\
    .groupBy("region")\
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"),2).alias("total_sales")
    )\
    .orderBy(desc("total_sales"))


In [0]:
gold_region_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_region_sales")

In [0]:
display(gold_region_sales.limit(10))

region,total_orders,total_quantity,total_sales
Online,2994,8844,3.5969582191E8
South,2615,7739,3.1661056784E8
North,2166,6523,2.6897290375E8
West,2100,6307,2.5802035052E8
East,1623,4826,1.9396488031E8
